# Klasifikasi Pendapatan (Adult Income) dengan Naive Bayes
Dataset: Adult / Census Income (UCI) – tabular, klasifikasi biner (`income`: `<=50K` vs `>50K`).

In [1]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, KBinsDiscretizer
from sklearn.naive_bayes import GaussianNB, CategoricalNB
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)


DATA_PATH = "adult.csv"
TARGET = "income"

### 1. Memanggil Data

In [2]:
df_raw = pd.read_csv(DATA_PATH)
df = df_raw.copy()

print("Ukuran data (baris, kolom):", df.shape)
print(df.head())

Ukuran data (baris, kolom): (32561, 15)
   age workclass  fnlwgt     education  education.num marital.status         occupation   relationship   race     sex  capital.gain  capital.loss  hours.per.week native.country income
0   90         ?   77053       HS-grad              9        Widowed                  ?  Not-in-family  White  Female             0          4356              40  United-States  <=50K
1   82   Private  132870       HS-grad              9        Widowed    Exec-managerial  Not-in-family  White  Female             0          4356              18  United-States  <=50K
2   66         ?  186061  Some-college             10        Widowed                  ?      Unmarried  Black  Female             0          4356              40  United-States  <=50K
3   54   Private  140359       7th-8th              4       Divorced  Machine-op-inspct      Unmarried  White  Female             0          3900              40  United-States  <=50K
4   41   Private  264663  Some-college  

### 2. Praproses
#### a. Info & tipe kolom

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             32561 non-null  int64 
 1   workclass       32561 non-null  object
 2   fnlwgt          32561 non-null  int64 
 3   education       32561 non-null  object
 4   education.num   32561 non-null  int64 
 5   marital.status  32561 non-null  object
 6   occupation      32561 non-null  object
 7   relationship    32561 non-null  object
 8   race            32561 non-null  object
 9   sex             32561 non-null  object
 10  capital.gain    32561 non-null  int64 
 11  capital.loss    32561 non-null  int64 
 12  hours.per.week  32561 non-null  int64 
 13  native.country  32561 non-null  object
 14  income          32561 non-null  object
dtypes: int64(6), object(9)
memory usage: 3.7+ MB


In [4]:
df.describe()

,age,fnlwgt,education.num,capital.gain,capital.loss,hours.per.week
count,32561.000000,3.256100e+04,32561.000000,32561.000000,32561.000000,32561.000000
mean,38.581647,1.897784e+05,10.080679,1077.648844,87.303830,40.437456
std,13.640433,1.055500e+05,2.572720,7385.292085,402.960219,12.347429
min,17.000000,1.228500e+04,1.000000,0.000000,0.000000,1.000000
25%,28.000000,1.178270e+05,9.000000,0.000000,0.000000,40.000000
50%,37.000000,1.783560e+05,10.000000,0.000000,0.000000,40.000000
75%,48.000000,2.370510e+05,12.000000,0.000000,0.000000,45.000000
max,90.000000,1.484705e+06,16.000000,99999.000000,4356.000000,99.000000


#### Koreksi Tipe

In [5]:
#merapikan nama kolom
df.columns = df.columns.str.replace(".", "_", regex=False)

#memastikankolom angka berisi angka semua (kalo ada teks aneh otomatis jadi NaN)
kolom_angka = ["age", "fnlwgt", "education_num", "capital_gain", "capital_loss", "hours_per_week"]
df[kolom_angka] = df[kolom_angka].apply(pd.to_numeric, errors="coerce")

#mendefinisikan tipe data kategori
tipe_kategori = {
    'workclass': 'category',
    'education': 'category',
    'marital_status': 'category',
    'occupation': 'category',
    'relationship': 'category',
    'race': 'category',
    'sex': 'category',
    'native_country': 'category',
    'income': 'category' # Ini adalah TARGET
}
df = df.astype(tipe_kategori)

#mendefinisikan daftar kolom numerik dan kategorik (dipakai di langkah selanjutnya)
num_cols = kolom_angka
cat_cols = [c for c in tipe_kategori if c != TARGET]

print("Tipe Data Setelah Dikoreksi:")
print(df.dtypes)

Tipe Data Setelah Dikoreksi:
age                  int64
workclass         category
fnlwgt               int64
education         category
education_num        int64
marital_status    category
occupation        category
relationship      category
race              category
sex               category
capital_gain         int64
capital_loss         int64
hours_per_week       int64
native_country    category
income            category
dtype: object


#### b. Missing values
Pada dataset ini missing value **tidak ditulis sebagai NaN**, melainkan sebagai tanda `?`.
Tanda tersebut diubah menjadi `NaN` agar terdeteksi, lalu ditangani.

In [6]:
#mengganti karakter "?" menjadi format NaN (Not a Number)
df = df.replace("?", np.nan)

print('\nMissing Value:')
print(df.isnull().sum())


Missing Value:
age                  0
workclass         1836
fnlwgt               0
education            0
education_num        0
marital_status       0
occupation        1843
relationship         0
race                 0
sex                  0
capital_gain         0
capital_loss         0
hours_per_week       0
native_country     583
income               0
dtype: int64


#### Penanganan Missing Value

In [7]:
#menghapus semua baris yang mengandung NaN
df = df.dropna()

#mengecek ukuran data dan sisa missing value
print(f"Ukuran data sekarang: {df.shape[0]} baris")
print('\nMissing Value setelah dropna:')
print(df.isnull().sum())

Ukuran data sekarang: 30162 baris

Missing Value setelah dropna:
age               0
workclass         0
fnlwgt            0
education         0
education_num     0
marital_status    0
occupation        0
relationship      0
race              0
sex               0
capital_gain      0
capital_loss      0
hours_per_week    0
native_country    0
income            0
dtype: int64


#### c. Duplikasi

In [8]:
#menghitung jumlah data yang duplikat
jumlah_duplikat = df.duplicated().sum()
print("Jumlah baris duplikat:", jumlah_duplikat)

#mnghapus baris yang duplikat dan mereset ulang nomor urut (index)
df = df.drop_duplicates().reset_index(drop=True)

#mengecek sisa baris dan kolom setelah dihapus
print("Ukuran data setelah hapus duplikat:", df.shape)

Jumlah baris duplikat: 23
Ukuran data setelah hapus duplikat: (30139, 15)


#### d.	Kategori (cek ejaan/kapitalisasi)

In [9]:
#menyamakan semua kolom kategorik dan target menjadi huruf kecil dan tanpa spasi di awal/akhir
for c in cat_cols + [TARGET]:
    df[c] = df[c].str.strip().str.lower()

for c in cat_cols + [TARGET]:
    print(f"{c} ({df[c].nunique()} nilai unik): {sorted(df[c].dropna().unique())}\n")

workclass (7 nilai unik): ['federal-gov', 'local-gov', 'private', 'self-emp-inc', 'self-emp-not-inc', 'state-gov', 'without-pay']

education (16 nilai unik): ['10th', '11th', '12th', '1st-4th', '5th-6th', '7th-8th', '9th', 'assoc-acdm', 'assoc-voc', 'bachelors', 'doctorate', 'hs-grad', 'masters', 'preschool', 'prof-school', 'some-college']

marital_status (7 nilai unik): ['divorced', 'married-af-spouse', 'married-civ-spouse', 'married-spouse-absent', 'never-married', 'separated', 'widowed']

occupation (14 nilai unik): ['adm-clerical', 'armed-forces', 'craft-repair', 'exec-managerial', 'farming-fishing', 'handlers-cleaners', 'machine-op-inspct', 'other-service', 'priv-house-serv', 'prof-specialty', 'protective-serv', 'sales', 'tech-support', 'transport-moving']

relationship (6 nilai unik): ['husband', 'not-in-family', 'other-relative', 'own-child', 'unmarried', 'wife']

race (5 nilai unik): ['amer-indian-eskimo', 'asian-pac-islander', 'black', 'other', 'white']

sex (2 nilai unik): ['

In [10]:
#penanganan kesalahan penulisan di kolom native_country
fix_country = {
    "holand-netherlands": "netherlands",
    "trinadad&tobago": "trinidad&tobago",
    "hong": "hong-kong",
    "columbia": "colombia",
}
df["native_country"] = df["native_country"].replace(fix_country)
print("Jumlah kategori native_country setelah perbaikan:", df["native_country"].nunique())
print(df[TARGET].value_counts())

Jumlah kategori native_country setelah perbaikan: 41
income
<=50k    22633
>50k      7506
Name: count, dtype: int64


### 3. Opsional
#### a. Outlier (IQR) pada kolom numerik penting

In [11]:
#mengecek outlier pakai metode IQR
kolom_cek = ["age", "hours_per_week", "capital_gain", "capital_loss"]

#membuat list kosong buat nampung hasil tiap kolom
hasil = []

for kolom in kolom_cek:
    # hitung Q1, Q3, dan IQR
    Q1 = df[kolom].quantile(0.25)
    Q3 = df[kolom].quantile(0.75)
    IQR = Q3 - Q1

    # batas normal data
    batas_bawah = Q1 - 1.5 * IQR
    batas_atas = Q3 + 1.5 * IQR

    # hitung data yang di luar batas = outlier
    jumlah_outlier = len(df[(df[kolom] < batas_bawah) | (df[kolom] > batas_atas)])
    persen = round(jumlah_outlier / len(df) * 100, 2)

    # simpan ke list
    hasil.append([kolom, batas_bawah, batas_atas, jumlah_outlier, persen])

#mengubah list jadi tabel
tabel_outlier = pd.DataFrame(hasil, columns=["kolom", "batas_bawah", "batas_atas", "jumlah_outlier", "persen"])
print(tabel_outlier)

            kolom  batas_bawah  batas_atas  jumlah_outlier  persen
0             age         -0.5        75.5             168    0.56
1  hours_per_week         32.5        52.5            7947   26.37
2    capital_gain          0.0         0.0            2538    8.42
3    capital_loss          0.0         0.0            1427    4.73


#### Penanganan Outlier

In [12]:
#penanganan outlier: nilai di luar batas diganti dengan nilai batasnya (clipping)
for kolom in ["age", "hours_per_week"]:
    Q1 = df[kolom].quantile(0.25)
    Q3 = df[kolom].quantile(0.75)
    IQR = Q3 - Q1
    batas_bawah = Q1 - 1.5 * IQR
    batas_atas = Q3 + 1.5 * IQR

    df[kolom] = df[kolom].clip(lower=batas_bawah, upper=batas_atas)

print(df[["age", "hours_per_week"]].describe())

                age  hours_per_week
count  30139.000000    30139.000000
mean      38.408673       41.420336
std       13.024807        6.146161
min       17.000000       32.500000
25%       28.000000       40.000000
50%       37.000000       40.000000
75%       47.000000       45.000000
max       75.500000       52.500000


#### b. Ketidakseimbangan kelas

In [13]:
counts = df[TARGET].value_counts()
print(counts)
print("--------------------------------")
print((counts / counts.sum() * 100).round(2).astype(str) + " %")

income
<=50k    22633
>50k      7506
Name: count, dtype: int64
--------------------------------
income
<=50k    75.1 %
>50k     24.9 %
Name: count, dtype: object


#### c. Indikasi kebocoran data (leakage)

In [14]:
#Memeriksa informasi yang sama atau berulang
check = pd.crosstab(df["education"], df["education_num"])
print("Education redundan:",
      (check.gt(0).sum(axis=1) == 1).all())

#Memeriksa korelasi fitur dengan target
tmp = df.copy()
tmp["y"] = (tmp[TARGET] == ">50k").astype(int)

print("\nKorelasi fitur dengan target:")
print(tmp[num_cols + ["y"]].corr()["y"].drop("y").round(2))

#memeriksa duplikasi fitur
print("\nJumlah baris dengan fitur identik:",
      df.drop(columns=[TARGET]).duplicated(keep=False).sum())

#menghapus informasi yang sama atau berulang dan fitur yang tidak digunakan
df = df.drop(columns=["education", "fnlwgt"], errors="ignore")
num_cols = [c for c in num_cols if c != "fnlwgt"]
cat_cols = [c for c in cat_cols if c != "education"]

Education redundan: True

Korelasi fitur dengan target:
age               0.24
fnlwgt           -0.01
education_num     0.34
capital_gain      0.22
capital_loss      0.15
hours_per_week    0.27
Name: y, dtype: float64

Jumlah baris dengan fitur identik: 23


### 4. Data hasil pembersihan: 5 teratas & 5 terbawah

In [15]:
print("=== 5 DATA TERATAS (setelah pembersihan) ===")
print(df.head(5))
print("\n=== 5 DATA TERBAWAH (setelah pembersihan) ===")
print(df.tail(5))

=== 5 DATA TERATAS (setelah pembersihan) ===
    age workclass  education_num marital_status         occupation   relationship   race     sex  capital_gain  capital_loss  hours_per_week native_country income
0  75.5   private              9        widowed    exec-managerial  not-in-family  white  female             0          4356            32.5  united-states  <=50k
1  54.0   private              4       divorced  machine-op-inspct      unmarried  white  female             0          3900            40.0  united-states  <=50k
2  41.0   private             10      separated     prof-specialty      own-child  white  female             0          3900            40.0  united-states  <=50k
3  34.0   private              9       divorced      other-service      unmarried  white  female             0          3770            45.0  united-states  <=50k
4  38.0   private              6      separated       adm-clerical      unmarried  white    male             0          3770            40.0

### 4. Encoding, split, normalisasi & standarisasi

#### a. Encoding

In [16]:
#proses encoding: ubah kolom kategorik (teks) jadi angka
# target dibuat angka dulu: <=50k = 0, >50k = 1
y = (df[TARGET] == ">50k").astype(int)

# fitur = semua kolom kecuali target
X = df.drop(columns=[TARGET]).copy()

# ubah tiap kolom teks jadi kode angka
for kolom in cat_cols:
    X[kolom] = X[kolom].astype("category").cat.codes

print("Contoh hasil encoding (5 data teratas):")
print(X.head())
print("\nTipe data setelah encoding:")
print(X.dtypes)

Contoh hasil encoding (5 data teratas):
    age  workclass  education_num  marital_status  occupation  relationship  race  sex  capital_gain  capital_loss  hours_per_week  native_country
0  75.5          2              9               6           3             1     4    0             0          4356            32.5              38
1  54.0          2              4               0           6             4     4    0             0          3900            40.0              38
2  41.0          2             10               5           9             3     4    0             0          3900            40.0              38
3  34.0          2              9               0           7             4     4    0             0          3770            45.0              38
4  38.0          2              6               5           0             4     4    1             0          3770            40.0              38

Tipe data setelah encoding:
age               float64
workclass            in

#### b. Membagi data training dan testing

In [17]:
# Bagi data: 80% training, 20% testing
# stratify=y supaya proporsi kelas di train dan test sama
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print("Data training :", X_train.shape)
print("Data testing  :", X_test.shape)

print("\nProporsi kelas data training:")
print(y_train.value_counts(normalize=True).round(3))
print("\nProporsi kelas data testing:")
print(y_test.value_counts(normalize=True).round(3))

Data training : (24111, 12)
Data testing  : (6028, 12)

Proporsi kelas data training:
income
0    0.751
1    0.249
Name: proportion, dtype: float64

Proporsi kelas data testing:
income
0    0.751
1    0.249
Name: proportion, dtype: float64


#### c. Normalisasi & standarisasi

In [18]:
# Normalisasi Min-Max (0-1) sebagai perbandingan
mm = MinMaxScaler()
X_train_mm = X_train.copy()
X_train_mm[num_cols] = mm.fit_transform(X_train[num_cols])

# Standarisasi (mean 0, std 1)
scaler = StandardScaler()
X_train_s, X_test_s = X_train.copy(), X_test.copy()
X_train_s[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test_s[num_cols] = scaler.transform(X_test[num_cols])

print("\nSetelah NORMALISASI Min-Max (min=0, max=1):")
print(X_train_mm[num_cols].describe().loc[["mean", "std", "min", "max"]].round(3))
print("Setelah STANDARISASI (mean~0, std~1):")
print(X_train_s[num_cols].describe().loc[["mean", "std", "min", "max"]].round(3))


Setelah NORMALISASI Min-Max (min=0, max=1):
        age  education_num  capital_gain  capital_loss  hours_per_week
mean  0.367          0.608         0.011         0.020           0.446
std   0.223          0.170         0.076         0.092           0.307
min   0.000          0.000         0.000         0.000           0.000
max   1.000          1.000         1.000         1.000           1.000
Setelah STANDARISASI (mean~0, std~1):
        age  education_num  capital_gain  capital_loss  hours_per_week
mean -0.000         -0.000        -0.000        -0.000           0.000
std   1.000          1.000         1.000         1.000           1.000
min  -1.644         -3.578        -0.148        -0.218          -1.450
max   2.839          2.305        13.066        10.602           1.804


### 5. Tampilkan 5 teratas & 5 terbawah data hasil praproses akhir
(kolom numerik sudah distandarisasi, kolom kategorik sudah menjadi kode angka)

In [19]:
final_train = X_train_s.copy()
final_train[TARGET] = y_train.map({0: "<=50k", 1: ">50k"})

print("=== 5 DATA TERATAS (hasil praproses akhir, data latih) ===")
print(final_train.head(5))
print("\n=== 5 DATA TERBAWAH (hasil praproses akhir, data latih) ===")
print(final_train.tail(5))

=== 5 DATA TERATAS (hasil praproses akhir, data latih) ===
            age  workclass  education_num  marital_status  occupation  relationship  race  sex  capital_gain  capital_loss  hours_per_week  native_country income
10521 -1.260459          2      -0.048105               4          12             3     4    1     -0.147593     -0.217634       -0.229745              38  <=50k
10894  0.808348          4      -0.048105               2           2             0     4    1     -0.147593     -0.217634        1.804371              30  <=50k
9731  -0.877346          2       2.305328               4           0             3     4    0     -0.147593     -0.217634       -0.229745              38  <=50k
7348   1.038215          0      -0.440343               0           0             1     4    0     -0.147593     -0.217634       -0.229745              38  <=50k
16967  1.421327          3      -0.048105               2          11             0     4    1     -0.147593     -0.217634        1

### 6. Model Naive Bayes & evaluasi
Dua varian dibandingkan:
1. **GaussianNB** pada fitur numerik terstandarisasi + kategorik one-hot (asumsi Gaussian kurang cocok untuk fitur biner).
2. **CategoricalNB** pada fitur kategorik + numerik yang didiskretisasi (binning) - cocok untuk data campuran seperti ini.

In [20]:
def evaluate(name, y_true, y_hat):
    print(f"===== {name} =====")
    print("Akurasi:", round(accuracy_score(y_true, y_hat), 4))
    print(classification_report(y_true, y_hat, target_names=["<=50k", ">50k"]))

# --- Model 1: GaussianNB (one-hot + standarisasi) ---
X_oh = pd.get_dummies(df.drop(columns=[TARGET]), columns=cat_cols, drop_first=True, dtype=int)
Xtr_oh, Xte_oh = X_oh.loc[X_train.index].copy(), X_oh.loc[X_test.index].copy()
sc2 = StandardScaler()
Xtr_oh[num_cols] = sc2.fit_transform(Xtr_oh[num_cols])
Xte_oh[num_cols] = sc2.transform(Xte_oh[num_cols])

gnb = GaussianNB().fit(Xtr_oh, y_train)
pred_gnb = gnb.predict(Xte_oh)
evaluate("GaussianNB (one-hot + standarisasi)", y_test, pred_gnb)

===== GaussianNB (one-hot + standarisasi) =====
Akurasi: 0.5629
              precision    recall  f1-score   support

       <=50k       0.97      0.43      0.60      4527
        >50k       0.36      0.96      0.52      1501

    accuracy                           0.56      6028
   macro avg       0.67      0.70      0.56      6028
weighted avg       0.82      0.56      0.58      6028



In [21]:
# --- Model 2: CategoricalNB (kategorik + numerik di-binning) ---
kb = KBinsDiscretizer(n_bins=8, encode="ordinal", strategy="quantile")
Xtr_c, Xte_c = X_train.copy(), X_test.copy()
Xtr_c[num_cols] = kb.fit_transform(X_train[num_cols]).astype(int)
Xte_c[num_cols] = kb.transform(X_test[num_cols]).astype(int)

# jumlah kategori minimum per fitur (agar kategori yang tidak muncul di data latih tidak error)
min_cats = [df[c].nunique() if c in cat_cols else 8 for c in Xtr_c.columns]

cnb = CategoricalNB(alpha=1.0, min_categories=min_cats).fit(Xtr_c, y_train)
pred_cnb = cnb.predict(Xte_c)
evaluate("CategoricalNB (kategorik + binning)", y_test, pred_cnb)

===== CategoricalNB (kategorik + binning) =====
Akurasi: 0.7946
              precision    recall  f1-score   support

       <=50k       0.91      0.80      0.85      4527
        >50k       0.56      0.77      0.65      1501

    accuracy                           0.79      6028
   macro avg       0.74      0.79      0.75      6028
weighted avg       0.83      0.79      0.80      6028

